# JHU Hub: scrape article content

Reads `./Data/JHU Hub Articles Links.csv`, requests every Hub URL, and extracts date, author, source, headline, subtitle, body, topics, and tags, keeping listing-page metadata as fallbacks.

Output: `./Data/JHU Hub Articles.csv`


In [ ]:
from pathlib import Path
from time import sleep
import html
import json
import random
import re
from urllib.parse import urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path("./Data")
LINKS_PATH = DATA_DIR / "JHU Hub Articles Links.csv"
OUTPUT_PATH = DATA_DIR / "JHU Hub Articles.csv"

REQUEST_TIMEOUT = 30
DELAY_RANGE = (0.8, 1.5)
CHECKPOINT_EVERY = 10

# Keep this True for a clean Johns Hopkins-owned corpus.
# Set it to False only if you deliberately want to scrape external publishers too.
SKIP_EXTERNAL_REDIRECTS = True

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
links = pd.read_csv(LINKS_PATH)
links = links.loc[:, ~links.columns.str.startswith("Unnamed:")]

required_columns = {"Link"}
missing_columns = required_columns - set(links.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in {LINKS_PATH}: {sorted(missing_columns)}")

links = links.drop_duplicates(subset="Link").reset_index(drop=True)
print(f"Loaded {len(links)} unique archive entries.")

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
def clean_text(value):
    """Decode HTML entities and collapse repeated whitespace."""
    if value is None:
        return ""
    return " ".join(html.unescape(str(value)).split())


def first_text(soup_or_tag, selectors):
    for selector in selectors:
        node = soup_or_tag.select_one(selector)
        if node:
            text = clean_text(node.get_text(" ", strip=True))
            if text:
                return text
    return ""


def first_attribute(soup_or_tag, selector_attribute_pairs):
    for selector, attribute in selector_attribute_pairs:
        node = soup_or_tag.select_one(selector)
        if node:
            value = clean_text(node.get(attribute, ""))
            if value:
                return value
    return ""


def iter_json_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_json_objects(child)


def get_article_json_ld(soup):
    """Find the most relevant Article/NewsArticle object in JSON-LD."""
    accepted_types = {
        "Article",
        "NewsArticle",
        "ReportageNewsArticle",
        "ScholarlyArticle",
    }

    for script in soup.select('script[type="application/ld+json"]'):
        raw = script.string or script.get_text()
        if not raw.strip():
            continue

        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            continue

        for obj in iter_json_objects(data):
            obj_type = obj.get("@type", "")
            obj_types = set(obj_type if isinstance(obj_type, list) else [obj_type])
            if obj_types & accepted_types:
                return obj

    return {}


def names_from_json_ld(value):
    if not value:
        return ""

    if isinstance(value, str):
        return clean_text(value)

    if isinstance(value, dict):
        return clean_text(value.get("name", ""))

    if isinstance(value, list):
        names = [names_from_json_ld(item) for item in value]
        return ", ".join(name for name in names if name)

    return ""


def normalize_byline(value):
    byline = clean_text(value)
    byline = re.sub(r"^\s*By\s+", "", byline, flags=re.IGNORECASE)
    return byline.strip(" /")


def is_hub_url(url):
    return urlsplit(url).netloc.lower() == "hub.jhu.edu"


def infer_source(url, soup, article_json):
    """Infer the publishing surface while preserving magazine articles."""
    path = urlsplit(url).path.lower()

    if path.startswith("/magazine/"):
        default_source = "Johns Hopkins Magazine"
    else:
        default_source = "Johns Hopkins Hub"

    source = names_from_json_ld(article_json.get("publisher"))
    if not source:
        source = first_attribute(
            soup,
            [
                ('meta[property="og:site_name"]', "content"),
                ('meta[name="application-name"]', "content"),
            ],
        )

    generic_values = {
        "",
        "Johns Hopkins University",
        "The Johns Hopkins University",
        "Hub",
    }

    return default_source if source in generic_values else source

In [ ]:
def locate_body_container(soup):
    """Return a likely article-body container and whether it is body-specific."""
    dedicated_selectors = [
        "[itemprop='articleBody']",
        "article .article-body",
        "article .article__body",
        "article .article-content",
        "article .article__content",
        "article .story-body",
        "article .entry-content",
        "main .article-body",
        "main .article__body",
        "main .article-content",
        "main .entry-content",
        ".article-body",
        ".article__body",
        ".entry-content",
    ]

    for selector in dedicated_selectors:
        node = soup.select_one(selector)
        if node:
            return node, True

    return soup.select_one("article") or soup.select_one("main"), False


def has_unwanted_ancestor(node):
    """Exclude captions, credits, sharing widgets, and related-content modules."""
    bad_fragments = {
        "caption",
        "credit",
        "share",
        "social",
        "related",
        "trending",
        "newsletter",
        "subscribe",
        "promo",
        "advert",
        "author-bio",
    }

    for parent in node.parents:
        if getattr(parent, "name", None) in {"article", "main", "body", "html"}:
            break

        classes = " ".join(parent.get("class", [])).lower()
        identifier = clean_text(parent.get("id", "")).lower()
        combined = f"{classes} {identifier}"

        if any(fragment in combined for fragment in bad_fragments):
            return True

    return False


def extract_body(soup, article_json, title="", subtitle="", author="", published_date=""):
    """Extract body paragraphs and section headings in document order."""
    container, body_specific = locate_body_container(soup)

    if container is None:
        return clean_text(article_json.get("articleBody", ""))

    body_soup = BeautifulSoup(str(container), "html.parser")

    for selector in [
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "form",
        "figure",
        "figcaption",
        "aside",
        "svg",
        "button",
        "[class*='share']",
        "[class*='social']",
        "[class*='related']",
        "[class*='trending']",
        "[class*='newsletter']",
        "[class*='subscribe']",
    ]:
        for node in body_soup.select(selector):
            node.decompose()

    stop_prefixes = (
        "posted in",
        "tagged ",
        "related content",
        "you might also like",
        "trending",
        "news network",
        "more from the ",
        "share on ",
        "more social media options",
    )

    metadata_values = {
        clean_text(title).casefold(),
        clean_text(subtitle).casefold(),
        clean_text(author).casefold(),
        clean_text(published_date).casefold(),
        f"by {clean_text(author)}".casefold() if author else "",
    }

    blocks = []
    started = body_specific or body_soup.find("h1") is None

    for node in body_soup.find_all(["h1", "h2", "h3", "h4", "p"]):
        if has_unwanted_ancestor(node):
            continue

        text = clean_text(node.get_text(" ", strip=True))
        if not text:
            continue

        lowered = text.casefold()

        if node.name == "h1":
            started = True
            continue

        if not started:
            continue

        if lowered.startswith(stop_prefixes):
            break

        if lowered in metadata_values:
            continue

        if re.match(r"^/?\s*published\s+", text, flags=re.IGNORECASE):
            continue

        if author and lowered == f"by {clean_text(author)}".casefold():
            continue

        # Header subtitles are usually the first h2 after the title.
        if subtitle and lowered == clean_text(subtitle).casefold():
            continue

        if not blocks or text != blocks[-1]:
            blocks.append(text)

    body = "\n\n".join(blocks).strip()
    return body or clean_text(article_json.get("articleBody", ""))


def extract_link_group(soup, path_fragment, lead_word):
    """Extract tags/topics, prioritizing the block introduced by Tagged/Posted in."""
    main = soup.select_one("article") or soup.select_one("main") or soup

    values = []
    for anchor in main.select(f'a[href*="{path_fragment}"]'):
        text = clean_text(anchor.get_text(" ", strip=True))
        if text and text not in values:
            values.append(text)

    # The broad selector is deliberate: JHU templates have changed over time.
    return "; ".join(values)

In [ ]:
def scrape_hub_article(soup, requested_url, final_url, listing_row):
    article_json = get_article_json_ld(soup)

    title = clean_text(article_json.get("headline", ""))
    if not title:
        title = first_text(soup, ["main h1", "article h1", "h1"])
    if not title:
        title = clean_text(listing_row.get("Article", ""))

    subtitle = first_text(
        soup,
        [
            "main h1 + h2",
            "article h1 + h2",
            "[class*='dek']",
            "[class*='deck']",
            "[class*='subtitle']",
            "[class*='subhead']",
            "[class*='description']",
        ],
    )
    if not subtitle:
        subtitle = clean_text(article_json.get("description", ""))
    if not subtitle:
        subtitle = first_attribute(
            soup,
            [
                ('meta[name="description"]', "content"),
                ('meta[property="og:description"]', "content"),
            ],
        )
    if not subtitle:
        subtitle = clean_text(listing_row.get("Listing Summary", ""))

    author = names_from_json_ld(article_json.get("author"))
    if not author:
        author = first_text(
            soup,
            [
                "[class*='byline']",
                "[class*='author-name']",
                "[rel='author']",
                "main p.author",
                "article p.author",
            ],
        )
    author = normalize_byline(author)

    published_date = clean_text(article_json.get("datePublished", ""))
    if not published_date:
        published_date = first_attribute(
            soup,
            [
                ('meta[property="article:published_time"]', "content"),
                ('meta[name="date"]', "content"),
                ("time[datetime]", "datetime"),
            ],
        )
    if not published_date:
        published_date = first_text(
            soup,
            [
                "[class*='published']",
                "[class*='date']",
                "main time",
                "article time",
            ],
        )
        published_date = re.sub(
            r"^/?\s*Published\s+",
            "",
            published_date,
            flags=re.IGNORECASE,
        )
    if not published_date:
        published_date = clean_text(listing_row.get("Listing Published Date", ""))

    source = infer_source(final_url, soup, article_json)

    body = extract_body(
        soup,
        article_json,
        title=title,
        subtitle=subtitle,
        author=author,
        published_date=published_date,
    )

    tags = extract_link_group(soup, "/tags/", "Tagged")
    topics = extract_link_group(soup, "/topics/", "Posted in")

    return {
        "Published Date": published_date,
        "Author": author,
        "Source": source,
        "Article Header": title,
        "Sub_Headings": subtitle,
        "Article Body": body,
        "Topics": topics,
        "Tags": tags,
        "Url": requested_url,
        "Final URL": final_url,
        "Listing Category": clean_text(listing_row.get("Listing Category", "")),
        "Listing Published Date": clean_text(
            listing_row.get("Listing Published Date", "")
        ),
        "Listing Summary": clean_text(listing_row.get("Listing Summary", "")),
    }

In [ ]:
def save_checkpoint(records):
    if not records:
        return

    frame = pd.DataFrame(records)
    frame = frame.drop_duplicates(subset="Url", keep="last").reset_index(drop=True)
    frame.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")


# Resume support.
records = []
completed_urls = set()

if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    existing = existing.loc[:, ~existing.columns.str.startswith("Unnamed:")]
    records = existing.to_dict("records")

    if "Url" in existing.columns:
        completed_urls = set(existing["Url"].dropna().astype(str))

    print(f"Resume mode: {len(completed_urls)} URLs already present in {OUTPUT_PATH}.")

In [ ]:
# -----------------------------
# Scrape all archive entries
# -----------------------------
pending_rows = links[~links["Link"].astype(str).isin(completed_urls)]
pending_records = pending_rows.to_dict("records")

for counter, listing_row in enumerate(
    tqdm(
        pending_records,
        total=len(pending_records),
        desc="JHU Hub articles",
    ),
    start=1,
):
    requested_url = str(listing_row["Link"])

    record = {
        "Published Date": clean_text(
            listing_row.get("Listing Published Date", "")
        ),
        "Author": "",
        "Source": "",
        "Article Header": clean_text(listing_row.get("Article", "")),
        "Sub_Headings": clean_text(listing_row.get("Listing Summary", "")),
        "Article Body": "",
        "Topics": "",
        "Tags": "",
        "Url": requested_url,
        "Final URL": "",
        "Listing Category": clean_text(
            listing_row.get("Listing Category", "")
        ),
        "Listing Published Date": clean_text(
            listing_row.get("Listing Published Date", "")
        ),
        "Listing Summary": clean_text(
            listing_row.get("Listing Summary", "")
        ),
        "External Redirect": False,
        "HTTP Status": "",
        "Error": "",
    }

    try:
        response = session.get(
            requested_url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )

        record["HTTP Status"] = response.status_code
        record["Final URL"] = response.url
        response.raise_for_status()

        external_redirect = not is_hub_url(response.url)
        record["External Redirect"] = external_redirect

        if external_redirect and SKIP_EXTERNAL_REDIRECTS:
            record["Source"] = urlsplit(response.url).netloc
            record["Error"] = (
                "External redirect skipped to keep the dataset limited "
                "to Johns Hopkins Hub/Johns Hopkins Magazine content."
            )
        else:
            soup = BeautifulSoup(response.text, "html.parser")
            record.update(
                scrape_hub_article(
                    soup=soup,
                    requested_url=requested_url,
                    final_url=response.url,
                    listing_row=listing_row,
                )
            )

            if not record["Article Header"]:
                record["Error"] = "Headline selector returned no value."
            elif not record["Article Body"]:
                record["Error"] = "Article body selector returned no value."

    except requests.RequestException as exc:
        record["Error"] = f"{type(exc).__name__}: {exc}"
    except Exception as exc:
        record["Error"] = f"{type(exc).__name__}: {exc}"

    records.append(record)

    if counter % CHECKPOINT_EVERY == 0:
        save_checkpoint(records)

    sleep(random.uniform(*DELAY_RANGE))

save_checkpoint(records)

final_count = len(pd.DataFrame(records).drop_duplicates(subset="Url"))
print(f"Saved {final_count} rows to: {OUTPUT_PATH.resolve()}")

In [ ]:
articles = pd.read_csv(OUTPUT_PATH)
articles.head()

In [ ]:
# Inspect failed, external, or incomplete rows.
problem_rows = articles[
    articles["Error"].fillna("").ne("")
    | articles["Article Header"].fillna("").eq("")
    | (
        ~articles["External Redirect"].fillna(False)
        & articles["Article Body"].fillna("").eq("")
    )
]

problem_rows[
    [
        "Article Header",
        "Url",
        "Final URL",
        "External Redirect",
        "HTTP Status",
        "Error",
    ]
].head(30)

In [ ]:
# Basic quality checks.
print("Duplicate URLs:", articles["Url"].duplicated().sum())
print(
    "External redirects:",
    articles["External Redirect"].fillna(False).sum(),
)
print(
    "Missing headlines:",
    articles["Article Header"].fillna("").eq("").sum(),
)
print(
    "Missing internal article bodies:",
    (
        ~articles["External Redirect"].fillna(False)
        & articles["Article Body"].fillna("").eq("")
    ).sum(),
)
print(
    "Rows with errors/status notes:",
    articles["Error"].fillna("").ne("").sum(),
)